In [ ]:
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_data, y_data, test_size=0.4, random_state=0)
lre.fit(x_train1[['horsepower']],y_train1)
lre.score(x_test1[['horsepower']],y_test1)


# sklearn.model_selection：sklearn 中负责模型选择与评估的子模块（之前用的 train_test_split 也来自这里）
# from ... import cross_val_score：导入交叉验证评分函数
from sklearn.model_selection import cross_val_score

# Rcross和上面比起来更慢，但是效果好
# lre 要评估的线性回归模型（之前创建的 lre = LinearRegression()）
# 特征：全部 201 辆车的马力（注意是 x_data 不是 x_train——交叉验证不需要你手动划分，它自己切）
# 目标：全部 201 个真实车价
# 切成 4 折（4-fold） 数据总共 201 行，每折约 50 个样本。
# 内部发生了什么
# 第1轮: 用第2,3,4份训练 → 在第1份上测 → R²_1
# 第2轮: 用第1,3,4份训练 → 在第2份上测 → R²_2
# 第3轮: 用第1,2,4份训练 → 在第3份上测 → R²_3
# 第4轮: 用第1,2,3份训练 → 在第4份上测 → R²_4
Rcross = cross_val_score(lre, x_data[['horsepower']], y_data, cv=4)
# Rcross 是一个含 4 个 R² 分数的数组  array([0.774, 0.514, 0.758, 0.714], dtype=float64)
Rcross
# mean（平均分）：比单次 train_test_split test_size=0.4 的分数更可靠的整体表现估计
# std（标准差）：波动大 → 模型表现不稳定，或数据分布不均匀
print("The mean of the folds are", Rcross.mean(), "and the standard deviation is" , Rcross.std())

# cross_val_score(lre, x_data[['horsepower']], y_data, cv=4, scoring='neg_mean_squared_error') 换个评价指标，不用R平方
# 用它时前面要乘 -1 取回真正的 MSE，返回的是负均方误差 注意：真正的 MSE 永远 ≥ 0 负号是sklearn 自己加的“包装”
# sklearn 的设计哲学是：所有评分指标统一“越大越好”；但 MSE 的性质是“越小越好”，和这个哲学冲突了。于是 sklearn 的做法是：返回 MSE 的相反数，并给它起名
# 预测质量	真实 MSE	    sklearn 返回的“负 MSE”
# 很好	    0.1	        -0.1（较大，接近 0）
# 很差	    500	        -500（较小）
-1 * cross_val_score(lre,x_data[['horsepower']], y_data,cv=4,scoring='neg_mean_squared_error')

